# recipe-dataclass — faded example 1: Recipe for a unary neg_forward

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `recipe-dataclass`. Running the beacon reports progress on the `Backprop: Recipe dataclass` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Recipe dataclass` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`recipe-dataclass`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "recipe-dataclass"
DD_SUBTOPIC = "Backprop: Recipe dataclass"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A `Recipe` packs the forward function, the raw positional args, the kwargs, and a `{argnum: parent}` dict. For a unary negation there is one Tensor input, so `parents` has a single entry keyed at argnum 0, `args` is a 1-tuple of the unboxed input, and `kwargs` is empty.

## Faded exercise 1

Complete `neg_forward(x)` for a `MiniTensor` input. Compute `out = -x.array`, wrap it, and attach a fully-populated `Recipe`. Fill in the `Recipe` construction so that `func` is `t.neg`, `args` is the 1-tuple of the unboxed input, `kwargs` is empty, and `parents` records the single Tensor parent at argnum 0.

**Fill in:** construct the Recipe with func, args, kwargs, and parents for the unary neg op

In [ ]:
from dataclasses import dataclass
from typing import Callable


class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = array
        self.recipe = recipe
        self.grad = None


@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict


def neg_forward(x: MiniTensor) -> MiniTensor:
    out = MiniTensor(-x.array)
    out.recipe = Recipe(
        func=t.neg,
        args=(x.array,),
        kwargs={},
        parents={0: x},
    )
    return out


x = MiniTensor(t.tensor([1.0, -2.0, 3.0]))
out = neg_forward(x)

def _test():
    xt = t.tensor([1.0, -2.0, 3.0])
    x = MiniTensor(xt)
    out = neg_forward(x)
    r = out.recipe
    assert isinstance(r, Recipe), 'recipe must be a Recipe instance'
    assert r.func is t.neg, 'func must be t.neg'
    assert isinstance(r.args, tuple) and len(r.args) == 1, 'args must be a 1-tuple'
    assert r.args[0] is xt, 'args[0] must be the unboxed input array'
    assert r.kwargs == {}, 'kwargs must be empty'
    assert set(r.parents.keys()) == {0}, 'parents must key only argnum 0'
    assert r.parents[0] is x, 'parent 0 must be the original MiniTensor x'
    assert t.allclose(out.array, -xt), 'forward value must be -x'

try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from dataclasses import dataclass
from typing import Callable


class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = array
        self.recipe = recipe
        self.grad = None


@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict


def neg_forward(x: MiniTensor) -> MiniTensor:
    out = MiniTensor(-x.array)
    out.recipe = Recipe(
        func=t.neg,
        args=(x.array,),
        kwargs={},
        parents={0: x},
    )
    return out


x = MiniTensor(t.tensor([1.0, -2.0, 3.0]))
out = neg_forward(x)
```
</details>